# Hedgers-Toll — a quantitative teardown 🔬
### Hedging-pressure IC · the long-short factor (HAC) · vs the commodity basket · the window sweep · the null

![Signal: Weak](https://img.shields.io/badge/Signal-Weak-dab617?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Premium today?: Faded](https://img.shields.io/badge/Premium_today%3F-Faded-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim with its standard error.* The steelman is §9.2: net hedger short-positioning (CFTC COT) predicts commodity returns. We prove the engine on a synthetic panel with a baked premium, then show it's absent on the modern CFTC-COT + futures tape.

> ⚠️ **Not investment advice.** The core executes on synthetic data; the real run is in [`../docs/results.md`](../docs/results.md), sources in [`../docs/references.md`](../docs/references.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back to intuition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../../.."))
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9.5, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from hedgers_toll import data, hedging, strategy, decompose, extension

# Offline synthetic commodities: a PREMIUM panel (hedging pressure predicts returns) and a NULL. The
# real CFTC-COT + futures verdict is in ../docs/results.md.
r,  hp,  truth = data.synthetic_commodities(hp_strength=0.0045, seed=29)   # the premium panel
r0, hp0, _     = data.synthetic_commodities(hp_strength=0.0,    seed=29)   # the null
print(f"{truth.n_comm} synthetic commodities x {truth.n_weeks} weeks | baked hp_strength={truth.hp_strength} | null=0")


12 synthetic commodities x 1040 weeks | baked hp_strength=0.0045 | null=0


## Beat 0 · Verdict

| Axis | Stamp | Why |
|---|---|---|
| **Signal** — does hedging pressure predict? | 🟡 `WEAK` | Strong on the control (IC HAC *t* > 11) and in the literature; on the modern tape IC **-0.020** (*t* **-1.4**). |
| **Tradability** | 🔴 `MIRAGE` | The long-short factor *loses*: Sharpe **-0.42** (*t* **-1.4**), negative across windows. |
| **Premium today?** | ⚪ `Faded` | Sub-period Sharpes **-0.79 / -0.99 / +0.60** — negative through most of 2015–2025. |

> **In one sentence:** the hedging-pressure / normal-backwardation premium is real in the long-run record and on our control, but absent at every parameter on the liquid commodity futures a trader faces today.

*(This notebook executes on the synthetic control; the real numbers are in [`../docs/results.md`](../docs/results.md).)*

## Beat 1 · The claim, precisely

$\text{HP}_i = (\text{CommShort}_i - \text{CommLong}_i)/(\text{CommShort}_i + \text{CommLong}_i)$ from the weekly COT; the tradable signal is its trailing z-score (lagged). The claim is $\mathbb{E}[r_{i,t+1}\mid \text{HP}_{i,t}\text{ high}] > 0$ — a long premium where hedgers are net short. The synthetic bakes $r_{t+1}=\text{hp\_strength}\cdot \text{HP}_t + \text{noise}$; $\text{hp\_strength}=0$ is the null.

In [2]:
pr = hedging.hedging_premium(r, hp)
print(f"top-minus-bottom tercile spread {pr['top_minus_bottom_ann_pct']:+.1f}%/yr (synthetic); "
      f"null {hedging.hedging_premium(r0, hp0)['top_minus_bottom_ann_pct']:+.1f}%/yr")

top-minus-bottom tercile spread +37.9%/yr (synthetic); null -3.4%/yr


## Beat 2 · So what?

Normal backwardation (Keynes-Hicks) and its modern cross-sectional form (Gorton-Hayashi-Rouwenhorst 2013; Basu-Miffre 2013) make hedging pressure a *structural* commodity risk premium — compensation for absorbing hedgers' inventory/price risk. The open questions: is the IC positive and significant, does the long-short factor pay net of cost, and is it stable across windows and time. Beats 4–6 answer all three; the answer here is no, no, and no.

## Beat 3 · Pre-registered protocol

1. **IC** (`hedging.hedging_premium`): pooled cross-sectional IC + *t*. Real ⇔ >0, |t|>2.
2. **Factor** (`decompose.premium_tstat`): long-short HAC *t*.
3. **vs basket** (`decompose.vs_basket`) + **window sweep** (`extension.window_sweep`).
4. **Null** collapses.

**Mirage line:** factor flat-to-negative on the real tape at every window.

## Beat 4 · The teardown

### 4a · IC and the factor, premium vs null

In [3]:
for label, (rr, hh) in [('premium', (r, hp)), ('null', (r0, hp0))]:
    pr = hedging.hedging_premium(rr, hh); pt = decompose.premium_tstat(rr, hh, cost_bps=10.0)
    print(f"{label:8s}: IC {pr['mean_ic']:+.3f} (t {pr['ic_t']:+.1f}); long-short {pt['mean_ann_pct']:+.1f}%/yr (HAC t {pt['t_stat']:+.1f})")

premium : IC +0.104 (t +11.2); long-short +17.2%/yr (HAC t +8.5)


null    : IC -0.002 (t -0.2); long-short -3.5%/yr (HAC t -1.7)


### 4b · The window sweep (on the synthetic premium)

In [4]:
sw = extension.window_sweep(r, hp, cost_bps=10.0)
display(sw.round(3))
print('On the REAL tape (../docs/results.md): the long-short Sharpe is NEGATIVE at every window (IC -0.020, factor -0.42).')

,ls_sharpe,ls_ann_return
window_weeks,,
52,1.7970,0.1540
104,2.0650,0.1780
156,2.0600,0.1720
208,2.0750,0.1740
260,2.1240,0.1790


On the REAL tape (../docs/results.md): the long-short Sharpe is NEGATIVE at every window (IC -0.020, factor -0.42).


> 💡 **In plain words.** On the synthetic, where the premium is real, the factor pays across every lookback. On the real commodities it loses at every lookback — so it isn't a tuning problem, the premium simply isn't being paid on this tape.

## Beat 5 · The verdict

- **Real on control** (4a): IC HAC *t* > 11.
- **Absent here** (real): IC -0.020 (*t* -1.4), factor Sharpe -0.42.
- **Not a window artefact** (4b): negative across windows.

> **Signal `WEAK` · Tradability `MIRAGE` · Premium today? `Faded`.**

## Beat 6 · Could you trade it?

- **No predictive signal** on the modern tape.
- **Likely crowded away** — every commodity risk-premia book reads the same COT.
- **May need breadth/history** the liquid-12, 10-year window lacks.

Tradability **`MIRAGE`**; premium today? **`Faded`**.

## Beat 7 · Going further

### 7a · Worked complement — robustness of the absence
The window sweep and the long-only-vs-long-short leg split on the real tape.

In [5]:
sw = extension.window_sweep(r, hp, cost_bps=10.0)
ls = extension.leg_split(r, hp, cost_bps=10.0)
print('synthetic premium -> factor positive across windows:', dict(sw['ls_sharpe'].round(2)))
print('Real tape (../docs/extension.md): negative across every window; long-only-top +0.18, basket +0.70.')

synthetic premium -> factor positive across windows: {52: np.float64(1.8), 104: np.float64(2.07), 156: np.float64(2.06), 208: np.float64(2.08), 260: np.float64(2.12)}
Real tape (../docs/extension.md): negative across every window; long-only-top +0.18, basket +0.70.


**The result.** On the real CFTC-COT + futures tape the hedging-pressure factor is negative at every signal window and in both legs — it is not a lookback that needs tuning, the premium is absent. Real in the long-run academic record and on our control; competed away (or specific to the broader, deeper samples the original studies used) on the liquid commodities a trader faces today. Full run in [`../docs/extension.md`](../docs/extension.md).

### 7b · Other forks
- **Disaggregated COT + breadth** — Producer/Merchant vs Managed Money over 30 contracts since the 1990s, where the academic premium lives.
- **The basis cousin (§9.1)** — term-structure roll yield is the other read on backwardation; does it pay where positioning doesn't?
- **Time-series, not cross-section** — own-commodity hedging pressure as a timing signal.

PRs welcome — widen the universe and history, or test the basis-based version.